In [1]:
# Instalações se necessário:
# !pip install requests beautifulsoup4 neo4j

import requests
from bs4 import BeautifulSoup
from neo4j import GraphDatabase
import time
import pandas as pd 

print("✅ Bibliotecas importadas com sucesso.")

✅ Bibliotecas importadas com sucesso.


In [3]:
# --- CONFIGURAÇÕES ---
URI = "neo4j://127.0.0.1:7687"
USER = "neo4j"
PASSWORD = "senha111"

def get_neo4j_driver():
    return GraphDatabase.driver(URI, auth=(USER, PASSWORD))

# Teste de conexão
try:
    driver = get_neo4j_driver()
    driver.verify_connectivity()
    print("✅ Conexão com Neo4j estabelecida!")
except Exception as e:
    print(f"❌ Erro na conexão: {e}")

✅ Conexão com Neo4j estabelecida!


In [1]:
import json
import os
import requests
from bs4 import BeautifulSoup

# 1. Configurações de pastas e arquivos
pasta_destino = 'Artigos_raw'
arquivo_links = 'links_artigos.txt'
arquivo_saida = os.path.join(pasta_destino, 'dados_artigos.json')

if not os.path.exists(pasta_destino):
    os.makedirs(pasta_destino)
    print(f"📁 Pasta '{pasta_destino}' criada.")

def extrair_dados(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status() # Garante que a requisição foi bem sucedida
        soup = BeautifulSoup(response.text, 'html.parser')

        # Título
        titulo_tag = soup.find("h1", class_="page_title")
        if not titulo_tag: return None
        titulo = titulo_tag.get_text(strip=True)

        # Data
        data_div = soup.find("div", class_="item published") or soup.find("section", class_="item published")
        data_pub = data_div.find("span", class_="value").get_text(strip=True) if data_div and data_div.find("span", class_="value") else "Data Desconhecida"

        # Volume/Edição
        volume_tag = soup.find("section", class_="item issue") or soup.find("div", class_="item issue")
        volume_info = volume_tag.find("a", class_="title").get_text(strip=True) if volume_tag and volume_tag.find("a", class_="title") else "Volume Indefinido"

        # Autores e Instituições
        autores_lista = []
        autores_tags = soup.select('ul.authors_list > li') or soup.select('.authors li')
        for li in autores_tags:
            nome_tag = li.find('span', class_='name') or li.find('strong')
            if nome_tag:
                nome = nome_tag.get_text(strip=True)
                inst_tag = li.find('span', class_='affiliation')
                inst = inst_tag.get_text(strip=True) if inst_tag else "Instituição não informada"
                autores_lista.append({"nome": nome, "inst": inst})

        # DOI
        doi_div = soup.find("div", class_="item doi") or soup.find("section", class_="item doi")
        doi = doi_div.find("a").get_text(strip=True).replace("https://doi.org/", "") if doi_div and doi_div.find("a") else "Sem DOI"

        # Keywords
        kw_div = soup.find("section", class_="item keywords") or soup.find("div", class_="item keywords")
        keywords = []
        if kw_div:
            kw_val = kw_div.find("span", class_="value")
            if kw_val:
                keywords = [k.strip().capitalize() for k in kw_val.get_text(strip=True).replace('.', '').split(',')]

        return {
            "titulo": titulo, "data": data_pub, "doi": doi, "volume": volume_info,
            "autores": autores_lista, "keywords": keywords, "url": url
        }
    except Exception as e:
        print(f"⚠️ Erro ao acessar {url}: {e}")
        return None

# --- Execução Principal ---

if __name__ == "__main__":
    # 2. Ler as URLs do arquivo .txt
    if not os.path.exists(arquivo_links):
        print(f"❌ Erro: O arquivo {arquivo_links} não foi encontrado!")
    else:
        with open(arquivo_links, 'r', encoding='utf-8') as f:
            # Lê as linhas e remove espaços/quebras de linha vazias
            urls = [linha.strip() for linha in f.readlines() if linha.strip()]

        resultados = []
        total = len(urls)

        print(f"🚀 Iniciando extração de {total} links...")

        for i, link in enumerate(urls, 1):
            print(f"[{i}/{total}] Processando: {link}")
            dados = extrair_dados(link)
            if dados:
                resultados.append(dados)

        # 3. Salvar todos os resultados no JSON
        if resultados:
            with open(arquivo_saida, 'w', encoding='utf-8') as f:
                json.dump(resultados, f, ensure_ascii=False, indent=4)
            print(f"\n✅ Concluído! {len(resultados)} artigos salvos em: {arquivo_saida}")
        else:
            print("\n⚠️ Nenhum dado foi extraído com sucesso.")

📁 Pasta 'Artigos_raw' criada.
🚀 Iniciando extração de 463 links...
[1/463] Processando: Vol.16.1
⚠️ Erro ao acessar Vol.16.1: Invalid URL 'Vol.16.1': No scheme supplied. Perhaps you meant https://Vol.16.1?
[2/463] Processando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/3383
[3/463] Processando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/3657
[4/463] Processando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4116
[5/463] Processando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4299
[6/463] Processando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4230
[7/463] Processando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4241
[8/463] Processando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4263
[9/463] Processando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4291
[10/463] Processando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4293
[11/463] Processand

In [6]:
import json
import os

# 1. Criar a pasta se não existir
if not os.path.exists('data_raw'):
    os.makedirs('data_raw')
    print("📁 Pasta 'data_raw' criada.")

def extrair_dados_brutos(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.encoding = response.apparent_encoding 
        soup = BeautifulSoup(response.text, 'html.parser')

        titulo_tag = soup.find("h1", class_="page_title")
        if not titulo_tag: return None
        
        # IMPORTANTE: Convertendo as tags do Soup para STRING para poder salvar no JSON
        return {
            "titulo": titulo_tag.get_text(strip=True),
            "data_html": str(soup.find("div", class_="item published") or soup.find("section", class_="item published")),
            "volume_html": str(soup.find("section", class_="item issue") or soup.find("div", class_="item issue")),
            "autores_html": [str(li) for li in (soup.select('ul.authors_list > li') or soup.select('.authors li'))],
            "doi_html": str(soup.find("div", class_="item doi") or soup.find("section", class_="item doi")),
            "kw_html": str(soup.find("section", class_="item keywords") or soup.find("div", class_="item keywords")),
            "url": url
        }
    except Exception as e:
        print(f"⚠️ Erro ao acessar {url}: {e}")
        return None

# --- EXECUÇÃO ---
dados_acumulados_brutos = []
with open("links_artigos.txt", "r") as f:
    todos_os_links = [l.strip() for l in f.readlines() if l.strip().startswith("http")]

for i, url in enumerate(todos_os_links):
    print(f"[{i+1}/{len(todos_os_links)}] Coletando: {url}")
    resultado = extrair_dados_brutos(url)
    if resultado:
        dados_acumulados_brutos.append(resultado)
    time.sleep(1)

# SALVANDO O ARQUIVO DE BACKUP
with open("data_raw/artigos_brutos.json", "w", encoding="utf-8") as f:
    json.dump(dados_acumulados_brutos, f, ensure_ascii=False, indent=4)

print(f"\n💾 Backup salvo em 'data_raw/artigos_brutos.json'!")

📁 Pasta 'data_raw' criada.
[1/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/3383
[2/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/3657
[3/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4116
[4/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4299
[5/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4230
[6/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4241
[7/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4263
[8/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4291
[9/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4293
[10/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4298
[11/415] Coletando: https://journals-sol.sbc.org.br/index.php/jidm/article/view/4303
[12/415] Coletando: https://journals-sol.sbc.or

In [ ]:
def padronizar_instituicao(nome_raw):
    if not nome_raw or "No affiliation declared" in nome_raw or "Instituição não informada" in nome_raw:
        return "N/A"
    
    texto = nome_raw.upper().strip()
    
    # Mapeamento baseado no seu JSON
    if any(x in texto for x in ["SÃO PAULO", "USP", "SAO PAULO", "ICMC"]):
        return "Universidade de São Paulo (USP)"
    if any(x in texto for x in ["MINAS GERAIS", "UFMG"]):
        return "Universidade Federal de Minas Gerais (UFMG)"
    if any(x in texto for x in ["FLUMINENSE", "UFF", "FEDERAL UNIVERSITY OF RIO DE JANEIRO"]):  
        if "FLUMINENSE" in texto or "UFF" in texto:
            return "Universidade Federal Fluminense (UFF)"
        if "RIO DE JANEIRO" in texto or "UFRJ" in texto or "COPPE":
            return "Universidade Federal do Rio de Janeiro (UFRJ)"
    if any(x in texto for x in ["CAMPINAS", "UNICAMP"]):
        return "Universidade Estadual de Campinas (UNICAMP)"
    if any(x in texto for x in ["CEARÁ", "CEARA", "UFC"]):
        return "Universidade Federal do Ceará (UFC)"
    
    return nome_raw.strip() # Retorna o original se não bater em nenhum padrão

def realizar_tratamento_completo(lista_bruta):
    lista_limpa = []
    
    for item in lista_bruta:
        # 1. Tratamento de Autores e Instituições
        autores_limpos = []
        for autor_li in item['autores_raw']:
            nome_tag = autor_li.find('span', class_='name') or autor_li.find('strong')
            inst_tag = autor_li.find('span', class_='affiliation')
            
            if nome_tag:
                inst_limpa = padronizar_instituicao(inst_tag.get_text(strip=True) if inst_tag else "")
                autores_limpos.append({
                    "nome": nome_tag.get_text(strip=True),
                    "instituicao": inst_limpa
                })

        # 2. Tratamento de DOI
        doi = "Sem DOI"
        if item['doi_raw'] and item['doi_raw'].find("a"):
            doi = item['doi_raw'].find("a").get_text(strip=True).replace("https://doi.org/", "")

        # 3. Montagem do dicionário final
        artigo_tratado = {
            "titulo": item['titulo'].strip(),
            "url": item['url'],
            "doi": doi,
            "autores": autores_limpos,
            "data": item['data_raw'].get_text(strip=True) if item['data_raw'] else "Data Desconhecida",
            "volume": item['volume_raw'].find("a", class_="title").get_text(strip=True) if item['volume_raw'] and item['volume_raw'].find("a", class_="title") else "Volume Indefinido"
        }
        lista_limpa.append(artigo_tratado)
        
    return lista_limpa

# Executa o tratamento
dados_prontos_para_neo4j = realizar_tratamento_completo(dados_acumulados_brutos)
print(f"✨ Dados tratados e padronizados! Exemplo do primeiro artigo:")
print(dados_prontos_para_neo4j[0])